# ***Imports & Notebook Setup***

In [134]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [135]:
pd.set_option('display.max_columns', None)  # Show all columns
"""
pd.set_option('display.max_rows', None)     # Show all rows (optional, careful if very large!)
pd.set_option('display.max_colwidth', None) # Optional: expand the column width
pd.set_option('display.width', 200) # Optional: display width to prevent wrapping
"""

"\npd.set_option('display.max_rows', None)     # Show all rows (optional, careful if very large!)\npd.set_option('display.max_colwidth', None) # Optional: expand the column width\npd.set_option('display.width', 200) # Optional: display width to prevent wrapping\n"

# ***mydata Import***

In [136]:
train_data = pd.read_csv('./data/train.csv')
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [137]:
test_data = pd.read_csv('./data/test.csv')
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


# ***Preprocessing Pipeline***

In [138]:
def preprocessing_pipeline(data, scaler, isTest=True):
    
    mydata = data.copy()
    RAW_FEATURES = mydata.columns.drop('Survived', errors='ignore').tolist()
    mydata = mydata[RAW_FEATURES + (['Survived'] if not isTest else [])]

    mydata['Title'] = mydata['Name'].apply(lambda x: x.split(',')[1].split()[0].strip().replace('.', ''))
    title_alias_map = {
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs",
        }

    titles = ['Mr', 'Mrs', 'Miss', 'Master', 'Dr', 'Rev']
    mydata['Title'] = mydata['Title'].replace(title_alias_map).apply(
        lambda x: x if x in titles else 'Rare'
        )
    
    title_age_imputation_map = mydata.groupby('Title')['Age'].mean()
    mydata.Age = mydata.Age.fillna(mydata.Title.map(title_age_imputation_map))

    mydata.Embarked = mydata.Embarked.fillna(mydata.Embarked.mode()[0])


    mydata['FamilySize'] = mydata['SibSp'] + mydata['Parch'] + 1
    mydata['TicketGroupSize'] = mydata.groupby('Ticket')['Ticket'].transform('count')
    mydata['IsAlone'] = (mydata['FamilySize'] == 1).astype(int)
    mydata['IsSoloTraveler'] = (mydata['TicketGroupSize'] == 1).astype(int)
    mydata['IsChild'] = (mydata['Age'] < 16).astype(int)
    mydata['FarePerPerson'] = mydata['Fare']/mydata['TicketGroupSize']


    mydata['Embarked'] = pd.Categorical(mydata['Embarked'], categories=['C', 'Q', 'S'])
    mydata = pd.get_dummies(mydata, columns=['Embarked'], prefix='Embarked', dtype=int)
    
    rare_titles = ['Dr', 'Rev', 'Rare'] 
    mydata['Title'] = mydata['Title'].apply(lambda x: 1 if x in rare_titles else 0)

    mydata['Sex'] = mydata['Sex'].map({'female': 0, 'male': 1})


    drop_cols = ['PassengerId', 'Cabin', 'Name', 'Ticket', 'SibSp', 'Parch']
    FINAL_FEATURES = [col for col in mydata.columns if col not in drop_cols]
    mydata = mydata[FINAL_FEATURES]
    
    
    scale_features = ['Age', 'Fare', 'FarePerPerson']
    if not isTest:
        survived = mydata.pop('Survived')
        mydata['Survived'] = survived

        scaler.fit(mydata[scale_features])
        
    mydata[scale_features] = scaler.transform(mydata[scale_features])

    return mydata, scaler


In [139]:
scaler = StandardScaler()

### ***Train Data***

In [140]:
final_train_data, scaler = preprocessing_pipeline(train_data, scaler, isTest=False)

In [141]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [142]:
final_train_data.head()

,Pclass,Sex,Age,Fare,Title,FamilySize,TicketGroupSize,IsAlone,IsSoloTraveler,IsChild,FarePerPerson,Embarked_C,Embarked_Q,Embarked_S,Survived
0,3,1,-0.584567,-0.502445,0,2,1,0,1,0,-0.496976,0,0,1,0
1,1,0,0.621430,0.786845,0,2,1,0,1,0,2.522573,1,0,0,1
2,3,0,-0.283068,-0.488854,0,1,1,1,1,0,-0.465145,0,0,1,1
3,1,0,0.395305,0.420730,0,2,2,0,0,0,0.413134,0,0,1,1
4,3,1,0.395305,-0.486337,0,1,1,1,1,0,-0.459251,0,0,1,0


### ***Test Data***

In [143]:
final_test_data, _ = preprocessing_pipeline(test_data, scaler, isTest=True)

In [144]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [145]:
final_test_data.head()

,Pclass,Sex,Age,Fare,Title,FamilySize,TicketGroupSize,IsAlone,IsSoloTraveler,IsChild,FarePerPerson,Embarked_C,Embarked_Q,Embarked_S
0,3,1,0.357618,-0.490783,0,1,1,1,1,0,-0.469663,0,1,0
1,3,0,1.299803,-0.507479,0,2,1,0,1,0,-0.508765,0,0,1
2,2,1,2.430424,-0.453367,0,1,1,1,1,0,-0.382033,0,1,0
3,3,1,-0.207693,-0.474005,0,1,1,1,1,0,-0.430368,0,0,1
4,3,0,-0.584567,-0.401017,0,3,1,0,1,0,-0.259428,0,0,1
